# Shared Linear Regression Baseline

Run Jupyter from the repository root. This example uses only the fixed temporal split and the listing-table features from `make_listing`; no auxiliary table is attached.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Start Jupyter from the repository root.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from sklearn.linear_model import LinearRegression

from src.evaluation import evaluate
from src.features import encode_high_card, make_listing, onehot_encoding, scale_numerical
from src.validation import load_data

## Load, clean, and split

`load_data` reads only `data/train.csv`, applies shared cleaning, removes exact duplicates, resets the cleaned index, and uses the fixed date ranges. The retained rows stay in their original order.

In [2]:
train_df, valid_df = load_data()

print(f"Training:   {len(train_df):,} rows, {train_df['RENT_APPROVAL_DATE'].min()} to {train_df['RENT_APPROVAL_DATE'].max()}")
print(f"Validation: {len(valid_df):,} rows, {valid_df['RENT_APPROVAL_DATE'].min()} to {valid_df['RENT_APPROVAL_DATE'].max()}")

Training:   117,777 rows, 2021-01 to 2024-04
Validation: 31,582 rows, 2024-05 to 2025-03


## Deterministic features

`make_listing` does not learn from data: it turns `RENT_APPROVAL_DATE` / `LEASE_COMMENCE_DATE` into `month_index` / `flat_age` and keeps the other listing columns. Imputation, scaling, and encoding do learn from data, so they are fitted on the training rows only.

In [3]:
X_train = make_listing(train_df)
X_valid = make_listing(valid_df)
y_train = train_df['MONTHLY_RENT']
y_valid = valid_df['MONTHLY_RENT']

assert list(X_train.columns) == list(X_valid.columns)
assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)
X_train.head()

,FLOOR_AREA_SQM,flat_age,month_index,TOWN,FLAT_TYPE,FLAT_MODEL,BLOCK,STREET
0,93.0,9.0,0,punggol,4 room,model a,604a,punggol road
1,110.0,20.0,0,sengkang,5 room,improved,290c,compassvale crescent
2,65.0,45.0,0,queenstown,3 room,improved,5,ghim moh road
3,96.0,11.0,0,sengkang,4 room,model a,436b,fernvale road
4,102.0,20.0,0,sembawang,4 room,premium apartment,467b,admiralty drive


## Fit and evaluate

Each step below is fitted on the first frame (training) and applied to the rest: median imputation + scaling for numeric columns, one-hot for low-card categoricals, ordinal for high-card categoricals. Replace the model to run a comparable experiment.

In [4]:
X_fit, X_eval = scale_numerical(X_train, X_valid)
X_fit, X_eval = onehot_encoding(X_fit, X_eval)
X_fit, X_eval = encode_high_card(X_fit, X_eval, method='ordinal')

model = LinearRegression().fit(X_fit, y_train)
predictions = model.predict(X_eval)

metrics = evaluate(y_valid, predictions)
assert all(np.isfinite(value) for value in metrics.values())
metrics

/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


{'RMSE': 615.6488197462302,
 'MAE': 459.1370653693951,
 'R2': 0.12645378062538482,
 'Mean Error': 351.75283050545954}